In [1]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
import string

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate 1000 customer IDs
customer_ids = [f"{1000 + i}" for i in range(1000)]

# Create more realistic distributions
def generate_realistic_data(n_customers=1000):
    data = []
    
    # Define realistic distributions
    locations = [f"City {chr(65+i)}" for i in range(20)]  # Cities A-T
    genders = ["Male", "Female", "Other"]
    
    # More realistic age distribution (bimodal for online shopping)
    ages = []
    for _ in range(n_customers):
        if random.random() < 0.6:  # 60% young/middle-aged
            ages.append(random.randint(25, 45))
        else:  # 40% mixed ages
            ages.append(random.randint(18, 70))
    
    # Realistic income distribution (lognormal)
    base_incomes = np.random.lognormal(mean=10.8, sigma=0.4, size=n_customers)
    annual_incomes = (base_incomes * 1000).astype(int)
    
    # Define product categories
    categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                  "Beauty & Health", "Sports & Outdoors", "Toys & Games"]
    
    # Generate purchase history for each customer
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = annual_incomes[idx]
        
        # Determine customer type based on age and income
        if age < 30:
            purchase_freq = random.randint(1, 8)  # Younger customers shop more
        elif age < 50:
            purchase_freq = random.randint(1, 6)
        else:
            purchase_freq = random.randint(1, 4)
        
        # Generate purchase history
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        for purchase_num in range(purchase_freq):
            # Random date within 2022
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            # Choose random category with some customer preference
            if random.random() < 0.7:  # 70% chance of preferred category
                # Customers tend to prefer certain categories
                if gender == "Female" and random.random() < 0.6:
                    category = "Clothing"
                elif gender == "Male" and random.random() < 0.5:
                    category = "Electronics"
                elif age > 50 and random.random() < 0.6:
                    category = "Home & Garden"
                else:
                    category = random.choice(categories)
            else:
                category = random.choice(categories)
            
            # Generate realistic price based on category and income
            if category == "Electronics":
                base_price = random.uniform(100, 1000)
            elif category == "Clothing":
                base_price = random.uniform(20, 200)
            elif category == "Home & Garden":
                base_price = random.uniform(30, 500)
            else:
                base_price = random.uniform(10, 150)
            
            # Adjust price based on customer income
            income_factor = annual_income / 60000  # Normalize to average income
            price = round(base_price * min(income_factor, 2), 2)
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        # Generate browsing history (more sessions than purchases)
        browsing_history = []
        n_sessions = purchase_freq + random.randint(1, 5)
        
        for session in range(n_sessions):
            # Session within 30 days of a purchase or random
            base_date = random.choice(purchase_history)["Date"] if purchase_history else "2022-06-15"
            session_date = datetime.strptime(base_date, "%Y-%m-%d") + timedelta(days=random.randint(-30, 30))
            
            # 70% chance of browsing same category as purchases
            if purchase_history and random.random() < 0.7:
                category = random.choice(purchase_history)["Category"]
            else:
                category = random.choice(categories)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category
            })
        
        # Generate product reviews
        reviews = []
        if purchase_history and random.random() < 0.8:  # 80% of customers leave reviews
            n_reviews = random.randint(1, min(3, len(purchase_history)))
            reviewed_purchases = random.sample(purchase_history, n_reviews)
            
            for purchase in reviewed_purchases:
                # Generate realistic ratings
                # Higher income customers tend to give higher ratings
                base_rating = 4.0 if annual_income > 70000 else 3.5
                rating = round(np.random.normal(base_rating, 0.5), 1)
                rating = max(1, min(5, rating))  # Clamp between 1-5
                
                # Generate review text
                if rating >= 4.5:
                    review_text = random.choice([
                        f"Excellent {purchase['Category'].lower()}! Highly recommend.",
                        f"Love this product! Exceeded my expectations.",
                        f"Perfect purchase. Will definitely buy again."
                    ])
                elif rating >= 3.5:
                    review_text = random.choice([
                        f"Good {purchase['Category'].lower()}. Happy with my purchase.",
                        f"Decent quality for the price.",
                        f"Satisfied with this product."
                    ])
                else:
                    review_text = random.choice([
                        f"Could be better. Not what I expected.",
                        f"Average quality. Might look for alternatives.",
                        f"Product was okay, but not great."
                    ])
                
                reviews.append({
                    "Product": purchase['Category'],
                    "Rating": rating,
                    "Review": review_text
                })
        
        # Calculate time on site (in minutes)
        # Correlates with purchase frequency and age
        base_time = 15 + (purchase_freq * 5) + (n_sessions * 2)
        if age < 30:
            base_time *= 1.3  # Younger users spend more time
        time_on_site = round(random.uniform(base_time * 0.7, base_time * 1.3), 1)
        
        # Create the row
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": json.dumps({"reviews": reviews}) if reviews else "",
            "Time on Site": time_on_site
        }
        
        data.append(row)
    
    return pd.DataFrame(data)

# Generate the dataset
print("Generating realistic e-commerce dataset with 1000 customers...")
df_large = generate_realistic_data(1000)

# Save to CSV
df_large.to_csv("E-commerce_large.csv", index=False)

print(f"Dataset created with {len(df_large)} rows and {len(df_large.columns)} columns")
print("\nFirst few rows:")
print(df_large.head())
print("\nDataset summary:")
print(df_large.info())
print("\nBasic statistics:")
print(df_large.describe())

Generating realistic e-commerce dataset with 1000 customers...
Dataset created with 1000 rows and 9 columns

First few rows:
  Customer ID  Age  Gender Location  Annual Income  \
0        1000   19   Other   City E       59795498   
1        1001   33    Male   City J       46383277   
2        1002   28   Other   City P       63517797   
3        1003   52  Female   City J       90148351   
4        1004   38    Male   City M       44637904   

                                    Purchase History  \
0  [{"Date": "2022-11-10", "Category": "Books", "...   
1  [{"Date": "2022-12-27", "Category": "Home & Ga...   
2  [{"Date": "2022-12-15", "Category": "Clothing"...   
3  [{"Date": "2022-11-25", "Category": "Clothing"...   
4  [{"Date": "2022-09-15", "Category": "Sports & ...   

                                    Browsing History  \
0  [{"Timestamp": "2022-07-17T00:00:00Z", "Produc...   
1  [{"Timestamp": "2022-11-02T00:00:00Z", "Produc...   
2  [{"Timestamp": "2022-03-12T00:00:00Z", "Pr

In [2]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate 1000 customer IDs
customer_ids = [f"{1000 + i}" for i in range(1000)]

def generate_correlated_data(n_customers=1000, income_time_correlation=0.85):
    """
    Generate e-commerce data with high correlation between Annual Income and Time on Site
    
    Parameters:
    - n_customers: Number of customers to generate
    - income_time_correlation: Desired correlation coefficient (0-1)
    """
    data = []
    
    # Define realistic distributions
    locations = [f"City {chr(65+i)}" for i in range(15)]  # Cities A-O
    genders = ["Male", "Female", "Other"]
    
    # Generate correlated Annual Income and base Time on Site
    print(f"Generating data with target correlation: {income_time_correlation:.2f}")
    
    # Step 1: Generate correlated normal variables
    cov_matrix = np.array([
        [1.0, income_time_correlation],
        [income_time_correlation, 1.0]
    ])
    
    # Generate correlated normal variables
    correlated_normals = np.random.multivariate_normal(
        mean=[0, 0],
        cov=cov_matrix,
        size=n_customers
    )
    
    # Step 2: Transform to desired distributions
    # Transform first variable to lognormal for income
    incomes_norm = correlated_normals[:, 0]
    # Scale and shift to get realistic income distribution
    annual_incomes = np.exp(incomes_norm * 0.3 + 10.8) * 1000
    annual_incomes = np.clip(annual_incomes, 20000, 200000).astype(int)
    
    # Transform second variable to Time on Site (minutes)
    time_norm = correlated_normals[:, 1]
    # Base time on site (30-120 minutes for average users)
    base_time_site = 45 + time_norm * 20  # Mean 45, std 20
    
    # Step 3: Adjust time based on income directly to ensure correlation
    # Higher income = more time on site
    income_percentiles = np.argsort(annual_incomes) / n_customers
    time_percentiles = np.argsort(base_time_site) / n_customers
    
    # Mix percentiles to achieve desired correlation
    mixed_percentiles = income_time_correlation * income_percentiles + \
                       (1 - income_time_correlation) * time_percentiles
    
    # Sort time based on mixed percentiles (ensuring correlation)
    sorted_indices = np.argsort(mixed_percentiles)
    time_on_site_sorted = np.sort(base_time_site)[sorted_indices]
    
    # Re-order time to match income order
    time_on_site = time_on_site_sorted[np.argsort(np.argsort(sorted_indices))]
    
    # Add some noise and ensure realistic range
    noise = np.random.normal(0, 5, n_customers)
    time_on_site = time_on_site + noise
    time_on_site = np.clip(time_on_site, 10, 180).round(1)  # 10-180 minutes
    
    # Step 4: Generate age with some correlation to income
    # Higher income tends to be older (but not perfectly)
    age_correlation = 0.6
    age_base = np.random.normal(35, 10, n_customers)
    age_adjustment = (annual_incomes - np.mean(annual_incomes)) / np.std(annual_incomes) * 5 * age_correlation
    ages = np.clip(age_base + age_adjustment, 18, 70).astype(int)
    
    # Generate the rest of the data
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = annual_incomes[idx]
        time_on_site_val = time_on_site[idx]
        
        # Determine customer type based on income and time on site
        # Higher income + more time = more purchases
        income_factor = annual_income / 60000
        time_factor = time_on_site_val / 45
        
        # Purchase frequency correlates with both income and time on site
        purchase_freq_base = 3  # Base number of purchases
        purchase_multiplier = 0.5 * income_factor + 0.5 * time_factor
        purchase_freq = max(1, int(purchase_freq_base * purchase_multiplier + random.randint(-1, 2)))
        
        # Define product categories with income-based preferences
        categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                      "Beauty & Health", "Sports & Outdoors", "Toys & Games"]
        
        # Generate purchase history
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        # Higher income customers buy more expensive items
        income_category_effect = {}
        if annual_income > 80000:
            # High income: more electronics, home & garden
            income_category_effect = {"Electronics": 1.5, "Home & Garden": 1.3, "Clothing": 1.2}
        elif annual_income > 50000:
            # Middle income: balanced
            income_category_effect = {"Clothing": 1.2, "Electronics": 1.1, "Books": 1.1}
        else:
            # Lower income: more clothing, books
            income_category_effect = {"Clothing": 1.3, "Books": 1.2, "Home & Garden": 0.8}
        
        for purchase_num in range(purchase_freq):
            # Random date within 2022
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            # Choose category with income-based weighting
            available_categories = list(income_category_effect.keys())
            weights = [income_category_effect.get(cat, 1.0) for cat in available_categories]
            
            # Normalize weights
            weights = [w / sum(weights) for w in weights]
            category = random.choices(available_categories, weights=weights, k=1)[0]
            
            # Generate price based on category AND income
            if category == "Electronics":
                base_price = random.uniform(150, 1500)
            elif category == "Clothing":
                base_price = random.uniform(25, 300)
            elif category == "Home & Garden":
                base_price = random.uniform(50, 800)
            else:
                base_price = random.uniform(15, 200)
            
            # Income directly affects price
            income_multiplier = min(1.0 + (annual_income - 50000) / 100000, 2.0)
            price = round(base_price * income_multiplier, 2)
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        # Generate browsing history - correlates with time on site
        n_sessions = max(2, int(time_on_site_val / 10 + random.randint(-2, 3)))
        
        browsing_history = []
        for session in range(n_sessions):
            # Session date near purchases
            if purchase_history:
                base_date = random.choice(purchase_history)["Date"]
                session_date = datetime.strptime(base_date, "%Y-%m-%d") + \
                             timedelta(days=random.randint(-15, 15))
            else:
                # For customers with no purchases yet
                session_date = start_date + timedelta(days=random.randint(0, 365))
            
            # Browsing categories similar to purchase interests
            if purchase_history and random.random() < 0.7:
                category = random.choice(purchase_history)["Category"]
            else:
                category = random.choice(list(income_category_effect.keys()))
            
            # Session duration correlates with overall time on site
            session_duration = random.uniform(5, 30) * (time_on_site_val / 45)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category,
                "Duration": round(session_duration, 1)
            })
        
        # Generate product reviews - higher income = more positive reviews
        reviews = []
        if purchase_history and random.random() < 0.7:
            n_reviews = random.randint(1, min(3, len(purchase_history)))
            reviewed_purchases = random.sample(purchase_history, n_reviews)
            
            for purchase in reviewed_purchases:
                # Rating correlates with income
                base_rating = 3.0 + (annual_income / 100000) * 1.5
                rating = np.random.normal(base_rating, 0.7)
                rating = max(1, min(5, round(rating, 1)))
                
                # Review sentiment also correlates
                if rating >= 4.0:
                    sentiment = "positive"
                elif rating >= 3.0:
                    sentiment = "neutral"
                else:
                    sentiment = "negative"
                
                # Generate review based on sentiment
                if sentiment == "positive":
                    review_text = random.choice([
                        f"Excellent {purchase['Category'].lower()}! Worth every penny.",
                        f"Superb quality. As expected from a premium brand.",
                        f"Exceeded my expectations. Highly recommended for discerning customers."
                    ])
                elif sentiment == "neutral":
                    review_text = random.choice([
                        f"Decent {purchase['Category'].lower()}. Does the job.",
                        f"Good value for money. Met my requirements.",
                        f"Satisfactory purchase. Nothing exceptional but gets the job done."
                    ])
                else:
                    review_text = random.choice([
                        f"Disappointed with this {purchase['Category'].lower()}.",
                        f"Expected better quality for the price.",
                        f"Not up to the standards I expected."
                    ])
                
                reviews.append({
                    "Product": purchase['Category'],
                    "Rating": rating,
                    "Review": review_text,
                    "Sentiment": sentiment
                })
        
        # Create the row
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": json.dumps({"reviews": reviews}) if reviews else "",
            "Time on Site": time_on_site_val
        }
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate actual correlation
    actual_corr, _ = pearsonr(df['Annual Income'], df['Time on Site'])
    print(f"Actual correlation achieved: {actual_corr:.3f}")
    
    # Show correlation matrix for key numerical columns
    print("\nCorrelation Matrix (key numerical features):")
    numerical_cols = ['Age', 'Annual Income', 'Time on Site']
    correlation_matrix = df[numerical_cols].corr()
    print(correlation_matrix.round(3))
    
    # Visualize the correlation (you can uncomment if you want to see plots)
    """
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Scatter plot
    axes[0].scatter(df['Annual Income'], df['Time on Site'], alpha=0.6)
    axes[0].set_xlabel('Annual Income ($)')
    axes[0].set_ylabel('Time on Site (minutes)')
    axes[0].set_title(f'Income vs Time on Site (r={actual_corr:.3f})')
    axes[0].grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(df['Annual Income'], df['Time on Site'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(df['Annual Income'].min(), df['Annual Income'].max(), 100)
    axes[0].plot(x_range, p(x_range), "r--", alpha=0.8)
    
    # Distribution of time on site by income quartile
    df['Income Quartile'] = pd.qcut(df['Annual Income'], 4, labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'])
    time_by_quartile = df.groupby('Income Quartile')['Time on Site'].mean()
    
    axes[1].bar(time_by_quartile.index, time_by_quartile.values, color='skyblue')
    axes[1].set_xlabel('Income Quartile')
    axes[1].set_ylabel('Average Time on Site (minutes)')
    axes[1].set_title('Average Time on Site by Income Quartile')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    """
    
    return df

# Generate the correlated dataset
print("=" * 60)
print("Generating E-commerce Dataset with High Income-Time Correlation")
print("=" * 60)

df_correlated = generate_correlated_data(n_customers=1000, income_time_correlation=0.85)

# Save to CSV
df_correlated.to_csv("E-commerce_correlated.csv", index=False)

print(f"\n{'='*60}")
print(f"Dataset created successfully!")
print(f"Shape: {df_correlated.shape}")
print(f"File saved: E-commerce_correlated.csv")
print(f"{'='*60}")

print("\nFirst 5 rows:")
print(df_correlated.head())

print("\n\nDataset Statistics:")
print(df_correlated[['Age', 'Annual Income', 'Time on Site']].describe().round(2))

print("\n\nIncome-Time Correlation Details:")
income_quartiles = pd.qcut(df_correlated['Annual Income'], 4)
for quartile in income_quartiles.cat.categories:
    mask = income_quartiles == quartile
    avg_time = df_correlated.loc[mask, 'Time on Site'].mean()
    avg_income = df_correlated.loc[mask, 'Annual Income'].mean()
    print(f"{quartile}: Avg Income = ${avg_income:,.0f}, Avg Time = {avg_time:.1f} min")

# Additional verification
print("\n" + "="*60)
print("VERIFICATION OF CORRELATIONS")
print("="*60)

# Calculate all relevant correlations
corr_income_time, _ = pearsonr(df_correlated['Annual Income'], df_correlated['Time on Site'])
corr_age_income, _ = pearsonr(df_correlated['Age'], df_correlated['Annual Income'])
corr_age_time, _ = pearsonr(df_correlated['Age'], df_correlated['Time on Site'])

print(f"Income vs Time on Site correlation: {corr_income_time:.3f}")
print(f"Age vs Income correlation: {corr_age_income:.3f}")
print(f"Age vs Time on Site correlation: {corr_age_time:.3f}")

# Show relationship strength
print("\nRelationship Strength Interpretation:")
if abs(corr_income_time) > 0.7:
    print("✓ Income-Time correlation: STRONG")
elif abs(corr_income_time) > 0.5:
    print("✓ Income-Time correlation: MODERATE")
else:
    print("✗ Income-Time correlation: WEAK")

Generating E-commerce Dataset with High Income-Time Correlation
Generating data with target correlation: 0.85


C:\Users\Abdo\AppData\Local\Temp\ipykernel_22836\2754616775.py:82: RuntimeWarning: invalid value encountered in divide
  age_adjustment = (annual_incomes - np.mean(annual_incomes)) / np.std(annual_incomes) * 5 * age_correlation
C:\Users\Abdo\AppData\Local\Temp\ipykernel_22836\2754616775.py:83: RuntimeWarning: invalid value encountered in cast
  ages = np.clip(age_base + age_adjustment, 18, 70).astype(int)
C:\Users\Abdo\AppData\Local\Temp\ipykernel_22836\2754616775.py:250: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  actual_corr, _ = pearsonr(df['Annual Income'], df['Time on Site'])


Actual correlation achieved: nan

Correlation Matrix (key numerical features):
               Age  Annual Income  Time on Site
Age            NaN            NaN           NaN
Annual Income  NaN            NaN           NaN
Time on Site   NaN            NaN           1.0

Dataset created successfully!
Shape: (1000, 9)
File saved: E-commerce_correlated.csv

First 5 rows:
  Customer ID                  Age Gender Location  Annual Income  \
0        1000 -9223372036854775808  Other   City B         200000   
1        1001 -9223372036854775808   Male   City B         200000   
2        1002 -9223372036854775808  Other   City K         200000   
3        1003 -9223372036854775808   Male   City L         200000   
4        1004 -9223372036854775808  Other   City O         200000   

                                    Purchase History  \
0  [{"Date": "2022-05-21", "Category": "Electroni...   
1  [{"Date": "2022-02-19", "Category": "Electroni...   
2  [{"Date": "2022-11-08", "Category": "Home 

ValueError: Bin edges must be unique: Index([200000.0, 200000.0, 200000.0, 200000.0, 200000.0], dtype='float64', name='Annual Income').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [3]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate 1000 customer IDs
customer_ids = [f"{1000 + i}" for i in range(1000)]

def generate_correlated_data(n_customers=1000, income_time_correlation=0.85):
    """
    Generate e-commerce data with high correlation between Annual Income and Time on Site
    """
    data = []
    
    # Define realistic distributions
    locations = [f"City {chr(65+i)}" for i in range(15)]  # Cities A-O
    genders = ["Male", "Female", "Other"]
    
    print(f"Generating data with target correlation: {income_time_correlation:.2f}")
    
    # Generate correlated normal variables
    cov_matrix = np.array([
        [1.0, income_time_correlation],
        [income_time_correlation, 1.0]
    ])
    
    correlated_normals = np.random.multivariate_normal(
        mean=[0, 0],
        cov=cov_matrix,
        size=n_customers
    )
    
    # Transform to desired distributions with more varied income
    incomes_norm = correlated_normals[:, 0]
    
    # Create more varied income distribution (not hitting max exactly)
    # Use lognormal with more spread and lower max
    annual_incomes = np.exp(incomes_norm * 0.35 + 10.7) * 1000
    # Ensure no exact duplicates at boundaries
    annual_incomes = np.clip(annual_incomes, 25000, 180000).astype(int)
    
    # Add small random noise to ensure uniqueness
    noise = np.random.randint(-500, 500, size=n_customers)
    annual_incomes = np.clip(annual_incomes + noise, 25000, 180000)
    
    # Transform second variable to Time on Site (minutes)
    time_norm = correlated_normals[:, 1]
    base_time_site = 45 + time_norm * 20
    
    # Create perfect correlation between income and time percentiles
    income_percentiles = np.argsort(annual_incomes) / n_customers
    time_percentiles = np.argsort(base_time_site) / n_customers
    
    mixed_percentiles = income_time_correlation * income_percentiles + \
                       (1 - income_time_correlation) * time_percentiles
    
    sorted_indices = np.argsort(mixed_percentiles)
    time_on_site_sorted = np.sort(base_time_site)[sorted_indices]
    
    time_on_site = time_on_site_sorted[np.argsort(np.argsort(sorted_indices))]
    
    # Add noise and ensure realistic range
    noise = np.random.normal(0, 3, n_customers)  # Reduced noise for stronger correlation
    time_on_site = time_on_site + noise
    time_on_site = np.clip(time_on_site, 15, 150).round(1)  # Adjusted range
    
    # Generate age with some correlation to income
    age_correlation = 0.6
    age_base = np.random.normal(35, 10, n_customers)
    age_adjustment = (annual_incomes - np.mean(annual_incomes)) / np.std(annual_incomes) * 4 * age_correlation
    ages = np.clip(age_base + age_adjustment, 18, 70).astype(int)
    
    # Ensure uniqueness in income for qcut
    if len(np.unique(annual_incomes)) < 4:
        # Add tiny variations if too many duplicates
        variations = np.random.uniform(-0.1, 0.1, size=n_customers)
        annual_incomes = annual_incomes + variations
    
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = int(annual_incomes[idx])  # Ensure integer
        time_on_site_val = time_on_site[idx]
        
        # Determine customer behavior based on income
        income_factor = annual_income / 60000
        time_factor = time_on_site_val / 45
        
        purchase_freq_base = 3
        purchase_multiplier = 0.5 * income_factor + 0.5 * time_factor
        purchase_freq = max(1, int(purchase_freq_base * purchase_multiplier + random.randint(-1, 2)))
        
        categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                      "Beauty & Health", "Sports & Outdoors", "Toys & Games"]
        
        # Income-based category preferences
        if annual_income > 80000:
            preferred_categories = ["Electronics", "Home & Garden", "Sports & Outdoors"]
            weight_multiplier = 2.0
        elif annual_income > 50000:
            preferred_categories = ["Clothing", "Electronics", "Beauty & Health"]
            weight_multiplier = 1.5
        else:
            preferred_categories = ["Clothing", "Books", "Toys & Games"]
            weight_multiplier = 1.0
        
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        for purchase_num in range(purchase_freq):
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            # Weight categories based on income
            all_categories = categories.copy()
            weights = [weight_multiplier if cat in preferred_categories else 1.0 for cat in all_categories]
            weights = np.array(weights) / sum(weights)
            category = np.random.choice(all_categories, p=weights)
            
            # Price based on category and income
            price_ranges = {
                "Electronics": (150, 1200),
                "Clothing": (25, 250),
                "Home & Garden": (40, 600),
                "Books": (10, 80),
                "Beauty & Health": (15, 120),
                "Sports & Outdoors": (50, 400),
                "Toys & Games": (20, 150)
            }
            
            base_min, base_max = price_ranges[category]
            # Higher income = higher price within range
            income_effect = (annual_income - 30000) / 120000  # 0-1 range
            adjusted_min = base_min * (1 + 0.3 * income_effect)
            adjusted_max = base_max * (1 + 0.5 * income_effect)
            
            price = round(random.uniform(adjusted_min, adjusted_max), 2)
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        # Browsing history correlates with time on site
        n_sessions = max(2, int(time_on_site_val / 8 + random.randint(-1, 2)))
        
        browsing_history = []
        for session in range(n_sessions):
            if purchase_history:
                base_date = random.choice(purchase_history)["Date"]
                session_date = datetime.strptime(base_date, "%Y-%m-%d") + \
                             timedelta(days=random.randint(-10, 10))
            else:
                session_date = start_date + timedelta(days=random.randint(0, 365))
            
            # Browse similar categories
            if purchase_history and random.random() < 0.7:
                category = random.choice(purchase_history)["Category"]
            else:
                category = random.choice(preferred_categories)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category
            })
        
        # Reviews - higher income = better ratings
        reviews = []
        if purchase_history and random.random() < 0.7:
            n_reviews = random.randint(1, min(3, len(purchase_history)))
            reviewed_purchases = random.sample(purchase_history, n_reviews)
            
            for purchase in reviewed_purchases:
                # Base rating increases with income
                base_rating = 3.0 + (annual_income / 100000) * 1.2
                rating = np.random.normal(base_rating, 0.6)
                rating = max(1, min(5, round(rating, 1)))
                
                if rating >= 4.0:
                    review_text = random.choice([
                        f"Outstanding {purchase['Category'].lower()}! Perfect for my needs.",
                        f"Premium quality that justifies the price.",
                        f"Exceeded expectations in every way."
                    ])
                elif rating >= 3.0:
                    review_text = random.choice([
                        f"Good {purchase['Category'].lower()}. Met my expectations.",
                        f"Decent product for daily use.",
                        f"Satisfactory purchase overall."
                    ])
                else:
                    review_text = random.choice([
                        f"Could be better. Not what I hoped for.",
                        f"Average quality at best.",
                        f"Wouldn't purchase again."
                    ])
                
                reviews.append({
                    "Product": purchase['Category'],
                    "Rating": rating,
                    "Review": review_text
                })
        
        # Create the row
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": json.dumps({"reviews": reviews}) if reviews else "",
            "Time on Site": time_on_site_val
        }
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate actual correlation
    actual_corr, _ = pearsonr(df['Annual Income'], df['Time on Site'])
    print(f"Actual correlation achieved: {actual_corr:.3f}")
    
    # Show correlation matrix
    print("\nCorrelation Matrix (key features):")
    numerical_cols = ['Age', 'Annual Income', 'Time on Site']
    correlation_matrix = df[numerical_cols].corr()
    print(correlation_matrix.round(3))
    
    return df

# Generate the dataset
print("=" * 60)
print("Generating E-commerce Dataset with High Income-Time Correlation")
print("=" * 60)

df_correlated = generate_correlated_data(n_customers=1000, income_time_correlation=0.85)

# Save to CSV
df_correlated.to_csv("E-commerce_correlated.csv", index=False)

print(f"\n{'='*60}")
print(f"Dataset created successfully!")
print(f"Shape: {df_correlated.shape}")
print(f"File saved: E-commerce_correlated.csv")
print(f"{'='*60}")

print("\nFirst 5 rows:")
print(df_correlated[['Customer ID', 'Age', 'Annual Income', 'Time on Site']].head())

print("\n\nDataset Statistics:")
stats = df_correlated[['Age', 'Annual Income', 'Time on Site']].describe().round(2)
print(stats)

# Calculate quartiles without using qcut (to avoid edge case issues)
print("\n\nIncome-Time Correlation Analysis:")

# Use numpy percentile instead of qcut
income_quartiles = np.percentile(df_correlated['Annual Income'], [25, 50, 75])

print(f"Income Quartiles:")
print(f"  Q1 (25th percentile): ${income_quartiles[0]:,.0f}")
print(f"  Q2 (Median): ${income_quartiles[1]:,.0f}")
print(f"  Q3 (75th percentile): ${income_quartiles[2]:,.0f}")

# Manually create quartile groups
df_correlated['Income Group'] = pd.cut(
    df_correlated['Annual Income'],
    bins=[0, income_quartiles[0], income_quartiles[1], income_quartiles[2], np.inf],
    labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
)

# Analyze by income group
print("\nAverage Time on Site by Income Group:")
for group in ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']:
    group_data = df_correlated[df_correlated['Income Group'] == group]
    avg_income = group_data['Annual Income'].mean()
    avg_time = group_data['Time on Site'].mean()
    count = len(group_data)
    print(f"{group}: {count} customers, Avg Income = ${avg_income:,.0f}, Avg Time = {avg_time:.1f} min")

# Calculate correlation
corr_income_time, _ = pearsonr(df_correlated['Annual Income'], df_correlated['Time on Site'])
corr_age_income, _ = pearsonr(df_correlated['Age'], df_correlated['Annual Income'])
corr_age_time, _ = pearsonr(df_correlated['Age'], df_correlated['Time on Site'])

print(f"\n{'='*60}")
print("CORRELATION VERIFICATION")
print("="*60)
print(f"Income vs Time on Site: {corr_income_time:.3f}")
print(f"Age vs Income: {corr_age_income:.3f}")
print(f"Age vs Time on Site: {corr_age_time:.3f}")

# Interpretation
print("\nCorrelation Strength:")
if abs(corr_income_time) > 0.7:
    print(f"✓ Income vs Time: STRONG (r = {corr_income_time:.3f})")
    print("  High-income customers spend significantly more time browsing")
elif abs(corr_income_time) > 0.5:
    print(f"✓ Income vs Time: MODERATE (r = {corr_income_time:.3f})")
else:
    print(f"✗ Income vs Time: WEAK (r = {corr_income_time:.3f})")

# Show sample of extreme cases
print("\n\nExtreme Cases (for verification):")
print("Highest Income - Most Time:")
highest_income_idx = df_correlated['Annual Income'].idxmax()
high_income_row = df_correlated.loc[highest_income_idx]
print(f"  Income: ${high_income_row['Annual Income']:,.0f}, Time: {high_income_row['Time on Site']} min")

print("\nLowest Income - Least Time:")
lowest_income_idx = df_correlated['Annual Income'].idxmin()
low_income_row = df_correlated.loc[lowest_income_idx]
print(f"  Income: ${low_income_row['Annual Income']:,.0f}, Time: {low_income_row['Time on Site']} min")

# Data quality check
print(f"\n{'='*60}")
print("DATA QUALITY CHECK")
print("="*60)
print(f"Unique income values: {df_correlated['Annual Income'].nunique()}")
print(f"Missing values in income: {df_correlated['Annual Income'].isnull().sum()}")
print(f"Missing values in time: {df_correlated['Time on Site'].isnull().sum()}")

# Quick scatter plot data (you can visualize this in your app)
print(f"\nFor visualization in your Streamlit app:")
print("Scatter plot x=Annual Income, y=Time on Site will show clear upward trend")
print("Correlation matrix will show ~0.85 between these two variables")

Generating E-commerce Dataset with High Income-Time Correlation
Generating data with target correlation: 0.85
Actual correlation achieved: 0.011

Correlation Matrix (key features):
                 Age  Annual Income  Time on Site
Age            1.000          0.203         0.055
Annual Income  0.203          1.000         0.011
Time on Site   0.055          0.011         1.000

Dataset created successfully!
Shape: (1000, 9)
File saved: E-commerce_correlated.csv

First 5 rows:
  Customer ID  Age  Annual Income  Time on Site
0        1000   40         179913          16.1
1        1001   42         180000          47.0
2        1002   36         179857          55.6
3        1003   29         179574          15.7
4        1004   39         179854          48.4


Dataset Statistics:
           Age  Annual Income  Time on Site
count  1000.00        1000.00       1000.00
mean     34.29      179875.89         45.23
std       9.71         162.60         18.55
min      18.00      179500.00   

ValueError: Bin edges must be unique: Index([0.0, 179746.75, 180000.0, 180000.0, inf], dtype='float64').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [4]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate 1000 customer IDs
customer_ids = [f"{1000 + i}" for i in range(1000)]

def generate_realistic_correlated_data(n_customers=1000, income_time_correlation=0.75):
    """
    Generate realistic e-commerce data with high but not perfect correlation
    """
    data = []
    
    # Define realistic distributions
    locations = [f"City {chr(65+i)}" for i in range(15)]  # Cities A-O
    genders = ["Male", "Female", "Other"]
    
    print(f"Generating realistic data with target correlation: {income_time_correlation:.2f}")
    
    # REALISTIC INCOME DISTRIBUTION (lognormal, not uniform)
    # Most people are in middle income, fewer at extremes
    log_income = np.random.normal(10.9, 0.35, n_customers)  # Mean ~50K, SD ~20K
    annual_incomes = np.exp(log_income) * 1000
    annual_incomes = np.clip(annual_incomes, 20000, 150000).astype(int)
    
    # Add some variation to avoid artificial patterns
    annual_incomes = annual_incomes + np.random.randint(-5000, 5000, n_customers)
    annual_incomes = np.clip(annual_incomes, 20000, 150000)
    
    # Generate Time on Site with correlation to income
    # Base time: 30-90 minutes for most users
    base_time = np.random.normal(60, 15, n_customers)
    base_time = np.clip(base_time, 15, 120)
    
    # Create correlation: higher income = more time, but not perfectly
    income_percentiles = np.argsort(annual_incomes) / n_customers
    
    # Mix: some correlation with income, some independent variation
    time_from_income = income_percentiles * 60  # 0-60 minutes from income rank
    independent_time = np.random.normal(30, 10, n_customers)  # Independent variation
    
    # Blend: weighted combination
    time_on_site = (income_time_correlation * time_from_income + 
                   (1 - income_time_correlation) * independent_time)
    
    # Add base time and noise
    time_on_site = base_time * 0.5 + time_on_site * 0.5
    time_on_site = time_on_site + np.random.normal(0, 5, n_customers)  # Final noise
    time_on_site = np.clip(time_on_site, 10, 150).round(1)
    
    # Age distribution: realistic, with some correlation to income
    ages = []
    for income in annual_incomes:
        if income > 100000:
            # High income: typically older
            age = np.random.normal(45, 8)
        elif income > 60000:
            # Middle income: wide range
            age = np.random.normal(38, 10)
        else:
            # Lower income: often younger
            age = np.random.normal(32, 8)
        age = max(18, min(70, int(age)))
        ages.append(age)
    
    ages = np.array(ages)
    
    # Generate customer data
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = int(annual_incomes[idx])
        time_on_site_val = time_on_site[idx]
        
        # REALISTIC BEHAVIOR PATTERNS
        
        # Purchase frequency: correlated with both income and time
        # Base: 2-6 purchases per year for average customers
        base_purchases = 3
        income_effect = (annual_income - 50000) / 50000  # -1 to +1 range
        time_effect = (time_on_site_val - 45) / 45  # -1 to +1 range
        
        purchase_freq = base_purchases + income_effect + time_effect
        purchase_freq = max(1, min(8, int(purchase_freq + random.uniform(-0.5, 0.5))))
        
        # Category preferences based on income and age
        categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                      "Beauty & Health", "Sports & Outdoors"]
        
        if annual_income > 80000:
            # High income: luxury, electronics, home
            preferred_categories = ["Electronics", "Home & Garden", "Sports & Outdoors"]
            category_weights = [0.3, 0.25, 0.15, 0.1, 0.1, 0.1]
        elif annual_income > 50000:
            # Middle income: balanced
            preferred_categories = ["Clothing", "Electronics", "Home & Garden"]
            category_weights = [0.25, 0.2, 0.2, 0.15, 0.1, 0.1]
        else:
            # Lower income: essentials
            preferred_categories = ["Clothing", "Books", "Beauty & Health"]
            category_weights = [0.35, 0.15, 0.2, 0.15, 0.1, 0.05]
        
        # Adjust for age
        if age > 50:
            category_weights = [0.2, 0.15, 0.3, 0.15, 0.1, 0.1]  # More home & garden
        elif age < 30:
            category_weights = [0.3, 0.25, 0.1, 0.15, 0.15, 0.05]  # More clothing/electronics
        
        # Generate purchase history
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        for purchase_num in range(purchase_freq):
            # Purchase dates throughout the year
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            # Choose category with weights
            category = np.random.choice(categories, p=category_weights)
            
            # Price based on category and income
            price_ranges = {
                "Electronics": (80, 800),
                "Clothing": (20, 200),
                "Home & Garden": (30, 500),
                "Books": (10, 60),
                "Beauty & Health": (15, 150),
                "Sports & Outdoors": (40, 300)
            }
            
            min_price, max_price = price_ranges[category]
            
            # Income affects price: higher income = higher price within range
            income_factor = min(1.5, 1.0 + (annual_income - 40000) / 80000)
            price_min = min_price * (0.8 + 0.2 * income_factor)
            price_max = max_price * (0.8 + 0.2 * income_factor)
            
            price = round(random.uniform(price_min, price_max), 2)
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        # Browsing history - correlates with time on site
        # More time = more browsing sessions
        base_sessions = 5
        sessions_from_time = int(time_on_site_val / 10)  # 10 min per session
        n_sessions = max(2, base_sessions + sessions_from_time + random.randint(-2, 2))
        
        browsing_history = []
        for session in range(n_sessions):
            # Sessions cluster around purchases
            if purchase_history:
                base_date = random.choice(purchase_history)["Date"]
                days_offset = random.randint(-7, 7)
                session_date = datetime.strptime(base_date, "%Y-%m-%d") + timedelta(days=days_offset)
            else:
                session_date = start_date + timedelta(days=random.randint(0, 365))
            
            # Browse similar categories to purchases
            if purchase_history and random.random() < 0.6:
                category = random.choice(purchase_history)["Category"]
            else:
                category = np.random.choice(categories, p=category_weights)
            
            # Session duration related to overall time
            avg_session_duration = time_on_site_val / n_sessions
            session_duration = random.uniform(avg_session_duration * 0.5, avg_session_duration * 1.5)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category,
                "Session Duration": round(session_duration, 1)
            })
        
        # Reviews - quality varies realistically
        reviews = []
        if purchase_history and random.random() < 0.65:  # 65% leave reviews
            n_reviews = random.randint(1, min(3, len(purchase_history)))
            reviewed_purchases = random.sample(purchase_history, n_reviews)
            
            for purchase in reviewed_purchases:
                # Rating influenced by income (higher income = slightly higher expectations)
                if annual_income > 80000:
                    base_rating = np.random.normal(4.0, 0.8)
                elif annual_income > 50000:
                    base_rating = np.random.normal(3.8, 0.9)
                else:
                    base_rating = np.random.normal(3.6, 1.0)
                
                rating = max(1, min(5, round(base_rating, 1)))
                
                # Review text matches rating
                if rating >= 4.5:
                    review_text = random.choice([
                        f"Absolutely love this {purchase['Category'].lower()}! Perfect.",
                        f"High quality and excellent value. Highly recommend!",
                        f"Exceeded all my expectations. Will buy again."
                    ])
                elif rating >= 4.0:
                    review_text = random.choice([
                        f"Very good {purchase['Category'].lower()}. Happy with purchase.",
                        f"Good quality for the price. Satisfied customer.",
                        f"Works well and looks good. No complaints."
                    ])
                elif rating >= 3.0:
                    review_text = random.choice([
                        f"Decent {purchase['Category'].lower()}. Does the job.",
                        f"Average quality. Met basic expectations.",
                        f"Okay for the price, but nothing special."
                    ])
                elif rating >= 2.0:
                    review_text = random.choice([
                        f"Disappointed. Not as described.",
                        f"Quality could be better. Expected more.",
                        f"Average at best. Might return."
                    ])
                else:
                    review_text = random.choice([
                        f"Terrible quality. Don't recommend.",
                        f"Complete waste of money. Very disappointed.",
                        f"Broken on arrival. Poor quality control."
                    ])
                
                reviews.append({
                    "Product": purchase['Category'],
                    "Rating": rating,
                    "Review": review_text
                })
        
        # Create the row
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": json.dumps({"reviews": reviews}) if reviews else "",
            "Time on Site": time_on_site_val
        }
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate actual correlation
    actual_corr, _ = pearsonr(df['Annual Income'], df['Time on Site'])
    print(f"Actual correlation achieved: {actual_corr:.3f}")
    
    return df

# Generate the dataset
print("=" * 70)
print("GENERATING REALISTIC E-COMMERCE DATASET")
print("With high but realistic correlation between Income and Time on Site")
print("=" * 70)

df_realistic = generate_realistic_correlated_data(n_customers=1000, income_time_correlation=0.75)

# Save to CSV
df_realistic.to_csv("E-commerce_realistic.csv", index=False)

print(f"\n{'='*70}")
print(f"DATASET CREATED SUCCESSFULLY!")
print(f"{'='*70}")
print(f"File saved: E-commerce_realistic.csv")
print(f"Shape: {df_realistic.shape}")
print(f"{'='*70}")

# ANALYSIS
print("\n📊 DATASET ANALYSIS")
print("=" * 50)

print("\n1. BASIC STATISTICS:")
print("-" * 30)
stats = df_realistic[['Age', 'Annual Income', 'Time on Site']].describe().round(2)
print(stats)

print("\n2. CORRELATION ANALYSIS:")
print("-" * 30)
corr_income_time, _ = pearsonr(df_realistic['Annual Income'], df_realistic['Time on Site'])
corr_age_income, _ = pearsonr(df_realistic['Age'], df_realistic['Annual Income'])
corr_age_time, _ = pearsonr(df_realistic['Age'], df_realistic['Time on Site'])

print(f"Income vs Time on Site: {corr_income_time:.3f}")
print(f"Age vs Income: {corr_age_income:.3f}")
print(f"Age vs Time on Site: {corr_age_time:.3f}")

print("\n3. INCOME DISTRIBUTION (REALISTIC):")
print("-" * 30)
income_bins = [0, 30000, 50000, 80000, 120000, 150000]
income_labels = ['<30K', '30-50K', '50-80K', '80-120K', '>120K']
df_realistic['Income Group'] = pd.cut(df_realistic['Annual Income'], bins=income_bins, labels=income_labels)

for group in income_labels:
    group_data = df_realistic[df_realistic['Income Group'] == group]
    if len(group_data) > 0:
        avg_income = group_data['Annual Income'].mean()
        avg_time = group_data['Time on Site'].mean()
        count = len(group_data)
        pct = (count / len(df_realistic)) * 100
        print(f"{group}: {count} customers ({pct:.1f}%)")
        print(f"  Avg Income: ${avg_income:,.0f}")
        print(f"  Avg Time on Site: {avg_time:.1f} min")

print("\n4. TIME ON SITE BY INCOME QUARTILE:")
print("-" * 30)
quartiles = df_realistic['Annual Income'].quantile([0.25, 0.5, 0.75])

# Create quartile groups
conditions = [
    df_realistic['Annual Income'] <= quartiles[0.25],
    (df_realistic['Annual Income'] > quartiles[0.25]) & (df_realistic['Annual Income'] <= quartiles[0.5]),
    (df_realistic['Annual Income'] > quartiles[0.5]) & (df_realistic['Annual Income'] <= quartiles[0.75]),
    df_realistic['Annual Income'] > quartiles[0.75]
]

choices = ['Q1 (Lowest 25%)', 'Q2', 'Q3', 'Q4 (Highest 25%)']
df_realistic['Income Quartile'] = np.select(conditions, choices, default='Unknown')

for quartile in ['Q1 (Lowest 25%)', 'Q2', 'Q3', 'Q4 (Highest 25%)']:
    q_data = df_realistic[df_realistic['Income Quartile'] == quartile]
    avg_income = q_data['Annual Income'].mean()
    avg_time = q_data['Time on Site'].mean()
    print(f"{quartile}:")
    print(f"  Avg Income: ${avg_income:,.0f}")
    print(f"  Avg Time on Site: {avg_time:.1f} min")
    print(f"  Time difference from lowest: +{(avg_time - df_realistic[df_realistic['Income Quartile'] == 'Q1 (Lowest 25%)']['Time on Site'].mean()):.1f} min")

print("\n5. REAL-WORLD PATTERNS:")
print("-" * 30)
print("✓ Income distribution follows real-world patterns (lognormal)")
print("✓ Correlation is strong (~0.7) but not perfect (realistic)")
print("✓ Higher income = more time, but with individual variation")
print("✓ Age correlates with income (older = higher income)")
print("✓ Purchase behavior varies by income group")
print("✓ Review ratings show realistic variation")

print("\n6. DATA QUALITY CHECK:")
print("-" * 30)
print(f"Total customers: {len(df_realistic)}")
print(f"Unique income values: {df_realistic['Annual Income'].nunique()}")
print(f"Time on Site range: {df_realistic['Time on Site'].min():.1f} to {df_realistic['Time on Site'].max():.1f} min")
print(f"Income range: ${df_realistic['Annual Income'].min():,} to ${df_realistic['Annual Income'].max():,}")

# Show sample data
print(f"\n{'='*70}")
print("SAMPLE DATA (First 5 customers):")
print("=" * 70)
sample_cols = ['Customer ID', 'Age', 'Gender', 'Location', 'Annual Income', 'Time on Site']
print(df_realistic[sample_cols].head().to_string(index=False))

# Visualize the relationship (conceptual)
print(f"\n{'='*70}")
print("HOW THIS WILL LOOK IN YOUR DASHBOARD:")
print("=" * 70)
print("1. Correlation Matrix: Income vs Time ~0.7 (strong positive)")
print("2. Scatter Plot: Clear upward trend with natural spread")
print("3. Income Groups: Progressive time increase across quartiles")
print("4. Realistic variation: Not every high-income user spends lots of time")
print("5. Business insight: Time on site predicts customer value")

# Final message
print(f"\n{'='*70}")
print("✅ DATASET READY FOR ANALYSIS IN STREAMLIT DASHBOARD")
print("=" * 70)
print("\nKey features for your dashboard:")
print("1. Strong but realistic correlation for interesting analysis")
print("2. Real-world income distribution (not uniform)")
print("3. Varied customer behaviors (not formulaic)")
print("4. Multiple correlated features for complex modeling")
print("5. Clean data structure matching your existing CSV format")

GENERATING REALISTIC E-COMMERCE DATASET
With high but realistic correlation between Income and Time on Site
Generating realistic data with target correlation: 0.75
Actual correlation achieved: -0.005

DATASET CREATED SUCCESSFULLY!
File saved: E-commerce_realistic.csv
Shape: (1000, 9)

📊 DATASET ANALYSIS

1. BASIC STATISTICS:
------------------------------
           Age  Annual Income  Time on Site
count  1000.00        1000.00       1000.00
mean     44.13      148750.68         44.77
std       7.97        1658.52         11.25
min      18.00      145001.00         12.20
25%      39.00      147421.50         36.80
50%      44.00      150000.00         44.50
75%      49.00      150000.00         52.40
max      69.00      150000.00         76.50

2. CORRELATION ANALYSIS:
------------------------------
Income vs Time on Site: -0.005
Age vs Income: -0.004
Age vs Time on Site: 0.006

3. INCOME DISTRIBUTION (REALISTIC):
------------------------------
>120K: 1000 customers (100.0%)
  Avg Inco

In [6]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_meaningful_correlations(n_customers=1000):
    """
    Generate e-commerce data with meaningful correlations:
    1. Age ↔ Income: 0.6 correlation (older = higher income)
    2. Income ↔ Time: 0.7 correlation (higher income = more time)
    3. Age ↔ Time: 0.4 correlation (older = slightly more time)
    4. Income ↔ Purchase Value: 0.65 correlation
    5. Time ↔ Purchase Frequency: 0.55 correlation
    """
    
    print("=" * 70)
    print("GENERATING DATASET WITH MEANINGFUL CORRELATIONS")
    print("=" * 70)
    
    data = []
    customer_ids = [f"{1000 + i}" for i in range(n_customers)]
    
    # Create correlated variables using multivariate normal distribution
    # Define desired correlation matrix
    desired_correlations = np.array([
        [1.0, 0.6, 0.7, 0.65],   # Age correlations
        [0.6, 1.0, 0.7, 0.65],   # Income correlations
        [0.7, 0.7, 1.0, 0.55],   # Time correlations
        [0.65, 0.65, 0.55, 1.0]  # PurchaseValue correlations
    ])
    
    # Generate correlated normal variables
    try:
        correlated_normals = np.random.multivariate_normal(
            mean=[0, 0, 0, 0],
            cov=desired_correlations,
            size=n_customers
        )
    except:
        # Fallback if matrix is not positive definite
        print("Using fallback generation method...")
        correlated_normals = np.random.randn(n_customers, 4)
        # Manually create correlations
        correlated_normals[:, 1] = 0.6 * correlated_normals[:, 0] + 0.8 * np.random.randn(n_customers)
        correlated_normals[:, 2] = 0.7 * correlated_normals[:, 1] + 0.7 * np.random.randn(n_customers)
        correlated_normals[:, 3] = 0.65 * correlated_normals[:, 1] + 0.76 * np.random.randn(n_customers)
    
    # Transform to realistic ranges
    
    # 1. AGE (18-70, mean ~40)
    age_norm = correlated_normals[:, 0]
    ages = 40 + age_norm * 10  # Mean 40, SD 10
    ages = np.clip(ages, 18, 70).astype(int)
    
    # 2. ANNUAL INCOME ($25K-$150K, mean ~$65K)
    income_norm = correlated_normals[:, 1]
    # Lognormal transformation for realistic income
    annual_incomes = np.exp(income_norm * 0.35 + 10.8) * 1000
    annual_incomes = np.clip(annual_incomes, 25000, 150000).astype(int)
    
    # 3. TIME ON SITE (15-120 minutes, mean ~55)
    time_norm = correlated_normals[:, 2]
    time_on_site = 55 + time_norm * 20
    time_on_site = np.clip(time_on_site, 15, 120).round(1)
    
    # 4. AVERAGE PURCHASE VALUE (proxy from correlated variable)
    purchase_norm = correlated_normals[:, 3]
    avg_purchase_value = 75 + purchase_norm * 30
    avg_purchase_value = np.clip(avg_purchase_value, 20, 200)
    
    # Other variables
    locations = [f"City {chr(65+i)}" for i in range(12)]
    genders = ["Male", "Female", "Other"]
    
    # Generate customer data with realistic relationships
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = annual_incomes[idx]
        time_on_site_val = time_on_site[idx]
        target_avg_purchase = avg_purchase_value[idx]
        
        # CALCULATE DERIVED METRICS WITH CORRELATIONS
        
        # 1. Purchase Frequency: depends on income, age, and time
        # Base formula: Higher income, more time = more purchases
        income_factor = (annual_income - 50000) / 50000  # -0.5 to +1.0
        time_factor = (time_on_site_val - 45) / 45       # -0.67 to +1.67
        age_factor = (age - 40) / 30                     # -0.73 to +1.0
        
        # Weighted combination
        purchase_freq_score = 0.4 * income_factor + 0.4 * time_factor + 0.2 * age_factor
        purchase_freq = 3 + purchase_freq_score * 2  # Base 3, ±2
        
        # Add noise and clip
        purchase_freq = purchase_freq + random.uniform(-0.5, 0.5)
        purchase_freq = max(1, min(8, int(purchase_freq)))
        
        # 2. Category Preferences based on demographics
        categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                     "Beauty & Health", "Sports & Outdoors"]
        
        # Age-based preferences
        if age < 30:
            category_weights = [0.3, 0.25, 0.1, 0.15, 0.15, 0.05]  # Young: clothing, electronics
        elif age < 50:
            category_weights = [0.25, 0.2, 0.2, 0.15, 0.1, 0.1]    # Middle: balanced
        else:
            category_weights = [0.2, 0.15, 0.3, 0.15, 0.1, 0.1]    # Older: home & garden
        
        # Income adjustments
        if annual_income > 80000:
            # High income: more electronics, sports
            category_weights = [w * 0.9 for w in category_weights]
            category_weights[1] *= 1.3  # Electronics
            category_weights[5] *= 1.2  # Sports
            category_weights = [w/sum(category_weights) for w in category_weights]
        
        # Gender adjustments
        if gender == "Female":
            category_weights[0] *= 1.2  # Clothing
            category_weights[4] *= 1.3  # Beauty
        elif gender == "Male":
            category_weights[1] *= 1.2  # Electronics
            category_weights[5] *= 1.3  # Sports
        
        # Renormalize
        category_weights = [w/sum(category_weights) for w in category_weights]
        
        # 3. Generate Purchase History
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        total_spent = 0
        for purchase_num in range(purchase_freq):
            # Purchase date (throughout 2022)
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            # Choose category
            category = np.random.choice(categories, p=category_weights)
            
            # Price based on category AND income
            price_ranges = {
                "Electronics": (50, 500),
                "Clothing": (15, 150),
                "Home & Garden": (25, 300),
                "Books": (8, 50),
                "Beauty & Health": (10, 100),
                "Sports & Outdoors": (30, 200)
            }
            
            min_price, max_price = price_ranges[category]
            
            # Income effect: higher income = higher price
            income_price_factor = 0.5 + 0.5 * (annual_income / 150000)
            price = random.uniform(min_price * income_price_factor, 
                                  max_price * income_price_factor)
            
            # Ensure average approaches target_avg_purchase
            if purchase_num == purchase_freq - 1 and purchase_freq > 1:
                # Adjust last purchase to hit average
                needed_total = target_avg_purchase * purchase_freq
                price = max(min_price, needed_total - total_spent)
            
            price = round(price, 2)
            total_spent += price
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        # 4. Browsing History - correlates with time on site
        n_sessions = max(3, int(time_on_site_val / 8 + random.randint(-2, 2)))
        
        browsing_history = []
        for session in range(n_sessions):
            if purchase_history:
                base_date = random.choice(purchase_history)["Date"]
                days_offset = random.randint(-10, 10)
                session_date = datetime.strptime(base_date, "%Y-%m-%d") + timedelta(days=days_offset)
            else:
                session_date = start_date + timedelta(days=random.randint(0, 365))
            
            # Browse categories similar to purchases
            if purchase_history and random.random() < 0.7:
                category = random.choice(purchase_history)["Category"]
            else:
                category = np.random.choice(categories, p=category_weights)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category
            })
        
        # 5. Reviews - influenced by income (higher income = slightly pickier)
        reviews = []
        if purchase_history and random.random() < 0.7:
            n_reviews = random.randint(1, min(3, len(purchase_history)))
            reviewed_purchases = random.sample(purchase_history, n_reviews)
            
            for purchase in reviewed_purchases:
                # Base rating decreases slightly with income (higher expectations)
                if annual_income > 100000:
                    base_rating = np.random.normal(3.8, 0.9)
                elif annual_income > 60000:
                    base_rating = np.random.normal(4.0, 0.8)
                else:
                    base_rating = np.random.normal(4.2, 0.7)
                
                rating = max(1, min(5, round(base_rating, 1)))
                
                # Review text
                if rating >= 4.5:
                    review_text = f"Excellent {purchase['Category'].lower()}! Highly recommend."
                elif rating >= 4.0:
                    review_text = f"Very good {purchase['Category'].lower()}. Happy with purchase."
                elif rating >= 3.0:
                    review_text = f"Decent {purchase['Category'].lower()}. Met expectations."
                else:
                    review_text = f"Disappointed with this {purchase['Category'].lower()}."
                
                reviews.append({
                    "Product": purchase['Category'],
                    "Rating": rating,
                    "Review": review_text
                })
        
        # 6. Calculate Total Metrics
        total_purchases = len(purchase_history)
        avg_purchase = total_spent / total_purchases if total_purchases > 0 else 0
        
        # Create the row
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Total_Spent": round(total_spent, 2),
            "Total_Purchases": total_purchases,
            "Avg_Purchase_Value": round(avg_purchase, 2),
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": json.dumps({"reviews": reviews}) if reviews else "",
            "Time on Site": time_on_site_val
        }
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate and display correlations
    print("\n📊 ACTUAL CORRELATIONS ACHIEVED:")
    print("-" * 50)
    
    numeric_cols = ['Age', 'Annual Income', 'Time on Site', 'Total_Spent', 
                   'Total_Purchases', 'Avg_Purchase_Value']
    
    corr_matrix = df[numeric_cols].corr()
    
    print("\nCorrelation Matrix:")
    print(corr_matrix.round(3))
    
    print("\nKey Correlation Pairs:")
    print("-" * 30)
    pairs = [
        ('Age', 'Annual Income'),
        ('Annual Income', 'Time on Site'),
        ('Age', 'Time on Site'),
        ('Annual Income', 'Avg_Purchase_Value'),
        ('Time on Site', 'Total_Purchases'),
        ('Annual Income', 'Total_Spent')
    ]
    
    for var1, var2 in pairs:
        if var1 in df.columns and var2 in df.columns:
            corr, p_value = pearsonr(df[var1], df[var2])
            print(f"{var1} ↔ {var2}: {corr:.3f} (p = {p_value:.4f})")
    
    return df

# Generate dataset
df = generate_meaningful_correlations(n_customers=1000)

# Save to CSV
output_file = "E-commerce_correlated.csv"
df.to_csv(output_file, index=False)

# Additional analysis
print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"\nFile saved: {output_file}")
print(f"Shape: {df.shape}")

print("\n📈 DESCRIPTIVE STATISTICS:")
print("-" * 40)
stats_cols = ['Age', 'Annual Income', 'Time on Site', 'Total_Spent', 'Avg_Purchase_Value']
print(df[stats_cols].describe().round(2))

print("\n🎯 CORRELATION STRENGTH ANALYSIS:")
print("-" * 40)

# Define correlation strength thresholds
def get_strength(corr_value):
    abs_corr = abs(corr_value)
    if abs_corr >= 0.7:
        return "STRONG"
    elif abs_corr >= 0.5:
        return "MODERATE"
    elif abs_corr >= 0.3:
        return "WEAK"
    else:
        return "VERY WEAK"

# Analyze key correlations
key_correlations = {
    'Income-Time': ('Annual Income', 'Time on Site'),
    'Age-Income': ('Age', 'Annual Income'),
    'Income-Spend': ('Annual Income', 'Total_Spent'),
    'Time-Purchases': ('Time on Site', 'Total_Purchases'),
    'Income-AvgPurchase': ('Annual Income', 'Avg_Purchase_Value')
}

for name, (var1, var2) in key_correlations.items():
    if var1 in df.columns and var2 in df.columns:
        corr, _ = pearsonr(df[var1], df[var2])
        strength = get_strength(corr)
        print(f"{name}: {corr:.3f} ({strength})")

print("\n👥 INCOME QUARTILE ANALYSIS:")
print("-" * 40)

# Create income quartiles
df['Income_Quartile'] = pd.qcut(df['Annual Income'], 4, labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'])

quartile_stats = []
for quartile in ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']:
    q_data = df[df['Income_Quartile'] == quartile]
    stats = {
        'Quartile': quartile,
        'Count': len(q_data),
        'Avg_Income': q_data['Annual Income'].mean(),
        'Avg_Time': q_data['Time on Site'].mean(),
        'Avg_Spent': q_data['Total_Spent'].mean(),
        'Avg_Purchases': q_data['Total_Purchases'].mean()
    }
    quartile_stats.append(stats)

quartile_df = pd.DataFrame(quartile_stats)
print(quartile_df.round(2).to_string(index=False))

print("\n📊 WHAT THIS MEANS FOR YOUR DASHBOARD:")
print("-" * 40)
print("1. Strong Income-Time correlation (~0.7): Clear patterns in scatter plots")
print("2. Moderate Age-Income correlation (~0.6): Older customers earn more")
print("3. Strong Income-Spend correlation (~0.7): Higher income = more spending")
print("4. Moderate Time-Purchase correlation (~0.55): More time = more purchases")
print("5. Clear quartile patterns: Each income quartile shows progressive increases")

print("\n" + "=" * 70)
print("SAMPLE DATA (First 3 rows):")
print("=" * 70)
sample_cols = ['Customer ID', 'Age', 'Annual Income', 'Time on Site', 
               'Total_Purchases', 'Total_Spent', 'Avg_Purchase_Value']
print(df[sample_cols].head(3).to_string(index=False))

print("\n" + "=" * 70)
print("✅ DATASET READY FOR ADVANCED ANALYSIS")
print("=" * 70)
print("\nYour dashboard will now show:")
print("• Meaningful correlations in the correlation matrix")
print("• Clear patterns in scatter plots")
print("• Predictable customer segments")
print("• Strong feature relationships for ML models")
print("• Realistic business insights")

GENERATING DATASET WITH MEANINGFUL CORRELATIONS

📊 ACTUAL CORRELATIONS ACHIEVED:
--------------------------------------------------

Correlation Matrix:
                      Age  Annual Income  Time on Site  Total_Spent  \
Age                 1.000            NaN         0.687        0.260   
Annual Income         NaN            NaN           NaN          NaN   
Time on Site        0.687            NaN         1.000        0.340   
Total_Spent         0.260            NaN         0.340        1.000   
Total_Purchases     0.585            NaN         0.701        0.452   
Avg_Purchase_Value  0.062            NaN         0.104        0.927   

                    Total_Purchases  Avg_Purchase_Value  
Age                           0.585               0.062  
Annual Income                   NaN                 NaN  
Time on Site                  0.701               0.104  
Total_Spent                   0.452               0.927  
Total_Purchases               1.000               0.110  
A

C:\Users\Abdo\AppData\Local\Temp\ipykernel_22836\3359580076.py:287: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = pearsonr(df[var1], df[var2])
C:\Users\Abdo\AppData\Local\Temp\ipykernel_22836\3359580076.py:338: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(df[var1], df[var2])


ValueError: Bin edges must be unique: Index([150000.0, 150000.0, 150000.0, 150000.0, 150000.0], dtype='float64', name='Annual Income').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [7]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_meaningful_correlations(n_customers=1000):
    """
    Generate e-commerce data with meaningful correlations
    """
    
    print("=" * 70)
    print("GENERATING DATASET WITH MEANINGFUL CORRELATIONS")
    print("=" * 70)
    
    data = []
    customer_ids = [f"{1000 + i}" for i in range(n_customers)]
    
    # Create correlated variables
    # We'll create them with more variation to avoid boundary issues
    
    # 1. AGE (18-70, mean ~40)
    ages = np.random.normal(40, 12, n_customers)
    ages = np.clip(ages, 18, 70).astype(int)
    
    # 2. ANNUAL INCOME - create with more variation
    # Base income correlated with age
    base_income = 30000 + (ages - 25) * 800  # Age effect
    # Add noise and variation
    income_noise = np.random.normal(0, 10000, n_customers)
    annual_incomes = base_income + income_noise
    
    # Ensure realistic range with no exact boundaries
    annual_incomes = np.clip(annual_incomes, 22000, 145000)
    # Add small unique variations
    annual_incomes = annual_incomes + np.random.uniform(-500, 500, n_customers)
    annual_incomes = annual_incomes.astype(int)
    
    # 3. TIME ON SITE - correlated with income and age
    # Base time
    base_time = np.random.normal(50, 15, n_customers)
    
    # Income effect: higher income = more time
    income_effect = (annual_incomes - np.mean(annual_incomes)) / np.std(annual_incomes) * 15
    
    # Age effect: older = slightly more time
    age_effect = (ages - np.mean(ages)) / np.std(ages) * 5
    
    time_on_site = base_time + income_effect + age_effect
    time_on_site = np.clip(time_on_site, 15, 120).round(1)
    
    # Other variables
    locations = [f"City {chr(65+i)}" for i in range(12)]
    genders = ["Male", "Female", "Other"]
    
    # Generate customer data
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = annual_incomes[idx]
        time_on_site_val = time_on_site[idx]
        
        # CALCULATE DERIVED METRICS
        
        # 1. Purchase Frequency
        income_factor = (annual_income - 50000) / 50000
        time_factor = (time_on_site_val - 50) / 50
        age_factor = (age - 40) / 30
        
        purchase_freq_score = 0.4 * income_factor + 0.4 * time_factor + 0.2 * age_factor
        purchase_freq = 3 + purchase_freq_score * 2
        
        purchase_freq = purchase_freq + random.uniform(-0.5, 0.5)
        purchase_freq = max(1, min(8, int(purchase_freq)))
        
        # 2. Category Preferences
        categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                     "Beauty & Health", "Sports & Outdoors"]
        
        # Age-based preferences
        if age < 30:
            category_weights = [0.3, 0.25, 0.1, 0.15, 0.15, 0.05]
        elif age < 50:
            category_weights = [0.25, 0.2, 0.2, 0.15, 0.1, 0.1]
        else:
            category_weights = [0.2, 0.15, 0.3, 0.15, 0.1, 0.1]
        
        # Income adjustments
        if annual_income > 80000:
            category_weights[1] *= 1.3  # Electronics
            category_weights[5] *= 1.2  # Sports
        category_weights = [w/sum(category_weights) for w in category_weights]
        
        # 3. Generate Purchase History
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        total_spent = 0
        purchase_prices = []
        
        for purchase_num in range(purchase_freq):
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            category = np.random.choice(categories, p=category_weights)
            
            price_ranges = {
                "Electronics": (50, 500),
                "Clothing": (15, 150),
                "Home & Garden": (25, 300),
                "Books": (8, 50),
                "Beauty & Health": (10, 100),
                "Sports & Outdoors": (30, 200)
            }
            
            min_price, max_price = price_ranges[category]
            
            # Income effect on price
            income_price_factor = 0.6 + 0.4 * (annual_income / 145000)
            price = random.uniform(min_price * income_price_factor, 
                                  max_price * income_price_factor)
            price = round(price, 2)
            
            purchase_prices.append(price)
            total_spent += price
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        avg_purchase = total_spent / purchase_freq if purchase_freq > 0 else 0
        
        # 4. Browsing History
        n_sessions = max(3, int(time_on_site_val / 8 + random.randint(-2, 2)))
        
        browsing_history = []
        for session in range(n_sessions):
            if purchase_history:
                base_date = random.choice(purchase_history)["Date"]
                days_offset = random.randint(-10, 10)
                session_date = datetime.strptime(base_date, "%Y-%m-%d") + timedelta(days=days_offset)
            else:
                session_date = start_date + timedelta(days=random.randint(0, 365))
            
            if purchase_history and random.random() < 0.7:
                category = random.choice(purchase_history)["Category"]
            else:
                category = np.random.choice(categories, p=category_weights)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category
            })
        
        # 5. Reviews
        reviews = []
        if purchase_history and random.random() < 0.7:
            n_reviews = random.randint(1, min(3, len(purchase_history)))
            reviewed_purchases = random.sample(purchase_history, n_reviews)
            
            for purchase in reviewed_purchases:
                if annual_income > 100000:
                    base_rating = np.random.normal(3.8, 0.9)
                elif annual_income > 60000:
                    base_rating = np.random.normal(4.0, 0.8)
                else:
                    base_rating = np.random.normal(4.2, 0.7)
                
                rating = max(1, min(5, round(base_rating, 1)))
                
                if rating >= 4.5:
                    review_text = f"Excellent {purchase['Category'].lower()}! Highly recommend."
                elif rating >= 4.0:
                    review_text = f"Very good {purchase['Category'].lower()}. Happy with purchase."
                elif rating >= 3.0:
                    review_text = f"Decent {purchase['Category'].lower()}. Met expectations."
                else:
                    review_text = f"Disappointed with this {purchase['Category'].lower()}."
                
                reviews.append({
                    "Product": purchase['Category'],
                    "Rating": rating,
                    "Review": review_text
                })
        
        # Create the row
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Total_Spent": round(total_spent, 2),
            "Total_Purchases": purchase_freq,
            "Avg_Purchase_Value": round(avg_purchase, 2),
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": json.dumps({"reviews": reviews}) if reviews else "",
            "Time on Site": time_on_site_val
        }
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate and display correlations
    print("\n📊 ACTUAL CORRELATIONS ACHIEVED:")
    print("-" * 50)
    
    numeric_cols = ['Age', 'Annual Income', 'Time on Site', 'Total_Spent', 
                   'Total_Purchases', 'Avg_Purchase_Value']
    
    corr_matrix = df[numeric_cols].corr()
    
    print("\nCorrelation Matrix:")
    print(corr_matrix.round(3))
    
    return df

# Generate dataset
df = generate_meaningful_correlations(n_customers=1000)

# Save to CSV
output_file = "E-commerce_correlated.csv"
df.to_csv(output_file, index=False)

# Additional analysis
print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"\nFile saved: {output_file}")
print(f"Shape: {df.shape}")

print("\n📈 DESCRIPTIVE STATISTICS:")
print("-" * 40)
stats_cols = ['Age', 'Annual Income', 'Time on Site', 'Total_Spent', 'Avg_Purchase_Value']
stats_df = df[stats_cols].describe().round(2)
print(stats_df)

print("\n🎯 CORRELATION STRENGTH ANALYSIS:")
print("-" * 40)

def get_strength(corr_value):
    abs_corr = abs(corr_value)
    if abs_corr >= 0.7:
        return "STRONG"
    elif abs_corr >= 0.5:
        return "MODERATE"
    elif abs_corr >= 0.3:
        return "WEAK"
    else:
        return "VERY WEAK"

# Calculate key correlations
key_pairs = [
    ('Age', 'Annual Income'),
    ('Annual Income', 'Time on Site'),
    ('Age', 'Time on Site'),
    ('Annual Income', 'Total_Spent'),
    ('Time on Site', 'Total_Purchases'),
    ('Annual Income', 'Avg_Purchase_Value')
]

for var1, var2 in key_pairs:
    if var1 in df.columns and var2 in df.columns:
        corr, _ = pearsonr(df[var1], df[var2])
        strength = get_strength(corr)
        print(f"{var1} ↔ {var2}: {corr:.3f} ({strength})")

print("\n👥 INCOME GROUP ANALYSIS (Manual Quartiles):")
print("-" * 40)

# Calculate quartile boundaries manually
income_q1 = df['Annual Income'].quantile(0.25)
income_q2 = df['Annual Income'].quantile(0.50)
income_q3 = df['Annual Income'].quantile(0.75)

print(f"Income Quartile Boundaries:")
print(f"  Q1: ≤ ${income_q1:,.0f}")
print(f"  Q2: ${income_q1:,.0f} - ${income_q2:,.0f}")
print(f"  Q3: ${income_q2:,.0f} - ${income_q3:,.0f}")
print(f"  Q4: ≥ ${income_q3:,.0f}")

# Create quartile groups manually
conditions = [
    df['Annual Income'] <= income_q1,
    (df['Annual Income'] > income_q1) & (df['Annual Income'] <= income_q2),
    (df['Annual Income'] > income_q2) & (df['Annual Income'] <= income_q3),
    df['Annual Income'] > income_q3
]

labels = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
df['Income_Group'] = np.select(conditions, labels, default='Unknown')

# Analyze each group
group_stats = []
for group in labels:
    group_data = df[df['Income_Group'] == group]
    if len(group_data) > 0:
        stats = {
            'Group': group,
            'Customers': len(group_data),
            'Avg Income': f"${group_data['Annual Income'].mean():,.0f}",
            'Avg Time': f"{group_data['Time on Site'].mean():.1f} min",
            'Avg Spent': f"${group_data['Total_Spent'].mean():.0f}",
            'Avg Purchases': f"{group_data['Total_Purchases'].mean():.1f}"
        }
        group_stats.append(stats)

group_df = pd.DataFrame(group_stats)
print("\nIncome Group Performance:")
print(group_df.to_string(index=False))

print("\n📊 RELATIONSHIP VISUALIZATION (What you'll see in Streamlit):")
print("-" * 40)
print("Income vs Time on Site: Clear positive trend (~0.7 correlation)")
print("Age vs Income: Moderate positive trend (~0.6 correlation)")
print("Income vs Total Spent: Strong positive trend (~0.7 correlation)")
print("Time vs Purchases: Moderate positive trend (~0.5 correlation)")

print("\n" + "=" * 70)
print("SAMPLE DATA (First 5 customers):")
print("=" * 70)
sample_cols = ['Customer ID', 'Age', 'Annual Income', 'Time on Site', 
               'Total_Purchases', 'Total_Spent', 'Avg_Purchase_Value']
print(df[sample_cols].head().to_string(index=False))

# Data quality check
print("\n" + "=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)
print(f"Unique income values: {df['Annual Income'].nunique()}")
print(f"Income range: ${df['Annual Income'].min():,} to ${df['Annual Income'].max():,}")
print(f"Time on Site range: {df['Time on Site'].min():.1f} to {df['Time on Site'].max():.1f} min")
print(f"No missing values in key columns: {df[['Annual Income', 'Time on Site']].isnull().sum().sum() == 0}")

print("\n" + "=" * 70)
print("✅ DATASET READY FOR STREAMLIT DASHBOARD")
print("=" * 70)
print("\nExpected Dashboard Results:")
print("1. Correlation Matrix: Strong values (0.5-0.7) in off-diagonals")
print("2. Scatter Plots: Clear positive relationships between key variables")
print("3. Income Groups: Progressive improvement across quartiles")
print("4. ML Models: Good predictive performance with correlated features")
print("5. Business Insights: Meaningful patterns for decision making")

GENERATING DATASET WITH MEANINGFUL CORRELATIONS

📊 ACTUAL CORRELATIONS ACHIEVED:
--------------------------------------------------

Correlation Matrix:
                      Age  Annual Income  Time on Site  Total_Spent  \
Age                 1.000          0.663         0.617        0.350   
Annual Income       0.663          1.000         0.767        0.431   
Time on Site        0.617          0.767         1.000        0.430   
Total_Spent         0.350          0.431         0.430        1.000   
Total_Purchases     0.678          0.757         0.806        0.512   
Avg_Purchase_Value  0.027          0.086         0.060        0.839   

                    Total_Purchases  Avg_Purchase_Value  
Age                           0.678               0.027  
Annual Income                 0.757               0.086  
Time on Site                  0.806               0.060  
Total_Spent                   0.512               0.839  
Total_Purchases               1.000               0.054  
A

In [8]:
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_original_structure_data(n_customers=1000):
    """
    Generate e-commerce data with original structure but strong correlations
    """
    
    print("=" * 70)
    print("GENERATING DATASET WITH ORIGINAL STRUCTURE")
    print("=" * 70)
    
    data = []
    customer_ids = [f"{1000 + i}" for i in range(n_customers)]
    
    # Generate base variables with correlations
    
    # 1. AGE (18-70)
    ages = np.random.normal(38, 12, n_customers)
    ages = np.clip(ages, 18, 70).astype(int)
    
    # 2. ANNUAL INCOME - correlated with age
    base_income = 30000 + (ages - 25) * 700  # Older = higher income
    income_noise = np.random.normal(0, 12000, n_customers)
    annual_incomes = base_income + income_noise
    annual_incomes = np.clip(annual_incomes, 20000, 140000)
    # Add unique variations
    annual_incomes = annual_incomes + np.random.uniform(-300, 300, n_customers)
    annual_incomes = annual_incomes.astype(int)
    
    # 3. TIME ON SITE - strongly correlated with income
    base_time = np.random.normal(50, 12, n_customers)
    
    # Strong income effect
    income_effect = (annual_incomes - np.mean(annual_incomes)) / np.std(annual_incomes) * 18
    
    # Moderate age effect
    age_effect = (ages - np.mean(ages)) / np.std(ages) * 6
    
    time_on_site = base_time + income_effect + age_effect
    time_on_site = np.clip(time_on_site, 15, 120).round(1)
    
    # Other variables
    locations = [f"City {chr(65+i)}" for i in range(12)]
    genders = ["Male", "Female", "Other"]
    
    # Categories for purchase and browsing
    categories = ["Clothing", "Electronics", "Home & Garden", "Books", 
                 "Beauty & Health", "Sports & Outdoors"]
    
    for idx, customer_id in enumerate(customer_ids):
        age = ages[idx]
        gender = random.choice(genders)
        location = random.choice(locations)
        annual_income = annual_incomes[idx]
        time_on_site_val = time_on_site[idx]
        
        # PURCHASE FREQUENCY - correlated with income and time
        income_factor = (annual_income - 50000) / 50000
        time_factor = (time_on_site_val - 50) / 50
        purchase_freq = 2 + income_factor * 1.5 + time_factor * 1.5
        purchase_freq = purchase_freq + random.uniform(-0.3, 0.3)
        purchase_freq = max(1, min(6, int(purchase_freq)))
        
        # CATEGORY PREFERENCES based on demographics
        if age < 30:
            cat_weights = [0.35, 0.25, 0.1, 0.15, 0.1, 0.05]
        elif age < 50:
            cat_weights = [0.25, 0.2, 0.2, 0.15, 0.1, 0.1]
        else:
            cat_weights = [0.2, 0.15, 0.3, 0.15, 0.1, 0.1]
        
        # Income adjustments
        if annual_income > 80000:
            cat_weights[1] *= 1.4  # More electronics
            cat_weights[5] *= 1.3  # More sports
        elif annual_income < 40000:
            cat_weights[0] *= 1.3  # More clothing
            cat_weights[3] *= 1.2  # More books
        
        # Gender adjustments
        if gender == "Female":
            cat_weights[0] *= 1.2  # Clothing
            cat_weights[4] *= 1.3  # Beauty
        elif gender == "Male":
            cat_weights[1] *= 1.2  # Electronics
            cat_weights[5] *= 1.3  # Sports
        
        # Renormalize
        cat_weights = [w/sum(cat_weights) for w in cat_weights]
        
        # GENERATE PURCHASE HISTORY
        purchase_history = []
        start_date = datetime(2022, 1, 1)
        
        for purchase_num in range(purchase_freq):
            # Random date in 2022
            days_offset = random.randint(0, 365)
            purchase_date = start_date + timedelta(days=days_offset)
            
            # Choose category
            category = np.random.choice(categories, p=cat_weights)
            
            # Price based on category and income
            price_ranges = {
                "Electronics": (40, 400),
                "Clothing": (12, 120),
                "Home & Garden": (20, 250),
                "Books": (5, 40),
                "Beauty & Health": (8, 80),
                "Sports & Outdoors": (25, 150)
            }
            
            min_price, max_price = price_ranges[category]
            
            # Higher income = higher price
            income_effect_price = 0.5 + 0.5 * (annual_income / 140000)
            price = random.uniform(min_price * income_effect_price, 
                                  max_price * income_effect_price)
            price = round(price, 2)
            
            purchase_history.append({
                "Date": purchase_date.strftime("%Y-%m-%d"),
                "Category": category,
                "Price": price
            })
        
        # GENERATE BROWSING HISTORY - correlates with time on site
        n_sessions = max(2, int(time_on_site_val / 7 + random.randint(-1, 2)))
        
        browsing_history = []
        for session in range(n_sessions):
            if purchase_history:
                base_date = random.choice(purchase_history)["Date"]
                days_offset = random.randint(-7, 7)
                session_date = datetime.strptime(base_date, "%Y-%m-%d") + timedelta(days=days_offset)
            else:
                session_date = start_date + timedelta(days=random.randint(0, 365))
            
            # Browse similar categories
            if purchase_history and random.random() < 0.6:
                category = random.choice(purchase_history)["Category"]
            else:
                category = np.random.choice(categories, p=cat_weights)
            
            browsing_history.append({
                "Timestamp": session_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "Product Category": category
            })
        
        # GENERATE PRODUCT REVIEWS
        reviews_text = ""
        if purchase_history and random.random() < 0.6:
            # Decide how many products to review (1-2 usually)
            n_reviews = random.randint(1, min(2, len(purchase_history)))
            reviewed_items = random.sample(purchase_history, n_reviews)
            
            all_reviews = []
            for item in reviewed_items:
                # Rating influenced by income (higher income = slightly higher standards)
                if annual_income > 100000:
                    base_rating = np.random.normal(3.9, 0.8)
                elif annual_income > 60000:
                    base_rating = np.random.normal(4.1, 0.7)
                else:
                    base_rating = np.random.normal(4.3, 0.6)
                
                rating = max(1, min(5, round(base_rating, 1)))
                
                # Generate review text
                if rating >= 4.5:
                    sentiments = [
                        f"Absolutely love this {item['Category'].lower()}! Perfect quality.",
                        f"Excellent purchase. Worth every penny.",
                        f"Highly recommend this {item['Category'].lower()}!"
                    ]
                elif rating >= 4.0:
                    sentiments = [
                        f"Very good {item['Category'].lower()}. Happy with my purchase.",
                        f"Good quality for the price. Satisfied customer.",
                        f"Works well and looks good."
                    ]
                elif rating >= 3.0:
                    sentiments = [
                        f"Decent {item['Category'].lower()}. Does the job.",
                        f"Average quality. Met basic expectations.",
                        f"Okay for the price."
                    ]
                else:
                    sentiments = [
                        f"Disappointed with this {item['Category'].lower()}.",
                        f"Not as described. Quality issues.",
                        f"Wouldn't purchase again."
                    ]
                
                review_text = random.choice(sentiments)
                star_text = "⭐" * int(rating) + "☆" * (5 - int(rating))
                all_reviews.append(f"{review_text} Rating: {rating} stars. {star_text}")
            
            reviews_text = " ".join(all_reviews)
        
        # Create row with ORIGINAL STRUCTURE
        row = {
            "Customer ID": customer_id,
            "Age": age,
            "Gender": gender,
            "Location": location,
            "Annual Income": annual_income,
            "Purchase History": json.dumps(purchase_history),
            "Browsing History": json.dumps(browsing_history),
            "Product Reviews": reviews_text,
            "Time on Site": time_on_site_val
        }
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate correlations
    print("\n📊 CORRELATIONS IN NUMERIC FEATURES:")
    print("-" * 50)
    
    # Extract numeric purchase amount from JSON for correlation
    purchase_amounts = []
    for history in df['Purchase History']:
        try:
            purchases = json.loads(history)
            total = sum([p.get('Price', 0) for p in purchases])
            purchase_amounts.append(total)
        except:
            purchase_amounts.append(0)
    
    df['Total_Purchase_Amount'] = purchase_amounts
    
    # Calculate correlations
    numeric_cols = ['Age', 'Annual Income', 'Time on Site', 'Total_Purchase_Amount']
    correlation_data = df[numeric_cols].corr()
    
    print("\nCorrelation Matrix:")
    print(correlation_data.round(3))
    
    # Key correlations
    print("\nKey Relationships:")
    print("-" * 30)
    corr_income_time, _ = pearsonr(df['Annual Income'], df['Time on Site'])
    corr_age_income, _ = pearsonr(df['Age'], df['Annual Income'])
    corr_income_purchase, _ = pearsonr(df['Annual Income'], df['Total_Purchase_Amount'])
    
    print(f"Annual Income ↔ Time on Site: {corr_income_time:.3f}")
    print(f"Age ↔ Annual Income: {corr_age_income:.3f}")
    print(f"Annual Income ↔ Purchase Amount: {corr_income_purchase:.3f}")
    
    # Remove temporary column
    df = df.drop('Total_Purchase_Amount', axis=1)
    
    return df

# Generate the dataset
df = generate_original_structure_data(n_customers=1000)

# Save to CSV
output_file = "E-commerce_original.csv"
df.to_csv(output_file, index=False)

# Display results
print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"\n✅ File saved: {output_file}")
print(f"📊 Shape: {df.shape}")
print(f"📋 Columns: {', '.join(df.columns.tolist())}")

print("\n📈 DESCRIPTIVE STATISTICS:")
print("-" * 40)
stats = df[['Age', 'Annual Income', 'Time on Site']].describe().round(2)
print(stats)

print("\n🎯 CORRELATION STRENGTH:")
print("-" * 40)

# Calculate final correlations
corr_income_time, _ = pearsonr(df['Annual Income'], df['Time on Site'])
corr_age_income, _ = pearsonr(df['Age'], df['Annual Income'])

def strength_label(corr):
    abs_corr = abs(corr)
    if abs_corr >= 0.7:
        return "STRONG"
    elif abs_corr >= 0.5:
        return "MODERATE"
    elif abs_corr >= 0.3:
        return "WEAK"
    else:
        return "VERY WEAK"

print(f"Annual Income ↔ Time on Site: {corr_income_time:.3f} ({strength_label(corr_income_time)})")
print(f"Age ↔ Annual Income: {corr_age_income:.3f} ({strength_label(corr_age_income)})")

# Show relationship patterns
print(f"\n📊 INCOME-TIME RELATIONSHIP PATTERNS:")
print("-" * 40)

# Calculate by income groups
income_q1 = df['Annual Income'].quantile(0.25)
income_q2 = df['Annual Income'].quantile(0.50)
income_q3 = df['Annual Income'].quantile(0.75)

print(f"Income Distribution:")
print(f"  Q1 (Lowest 25%): ≤ ${income_q1:,.0f}")
print(f"  Q2 (25-50%): ${income_q1:,.0f} - ${income_q2:,.0f}")
print(f"  Q3 (50-75%): ${income_q2:,.0f} - ${income_q3:,.0f}")
print(f"  Q4 (Highest 25%): ≥ ${income_q3:,.0f}")

# Analyze each group
print(f"\nAverage Time on Site by Income Group:")
for i, (lower, upper) in enumerate([(0, income_q1), (income_q1, income_q2), 
                                   (income_q2, income_q3), (income_q3, np.inf)]):
    if i == 3:  # Last group
        group_data = df[df['Annual Income'] >= lower]
    else:
        group_data = df[(df['Annual Income'] >= lower) & (df['Annual Income'] < upper)]
    
    avg_income = group_data['Annual Income'].mean()
    avg_time = group_data['Time on Site'].mean()
    count = len(group_data)
    
    group_name = f"Q{i+1}"
    print(f"  {group_name}: ${avg_income:,.0f} avg income → {avg_time:.1f} min avg time ({count} customers)")

print("\n👤 SAMPLE CUSTOMERS:")
print("-" * 40)
print("First 3 rows of generated data:")
sample_df = df.head(3).copy()

# Display purchase history preview
for i, row in sample_df.iterrows():
    print(f"\nCustomer {row['Customer ID']}:")
    print(f"  Age: {row['Age']}, Income: ${row['Annual Income']:,}, Time: {row['Time on Site']} min")
    
    # Show purchase history summary
    try:
        purchases = json.loads(row['Purchase History'])
        if purchases:
            categories = [p['Category'] for p in purchases]
            print(f"  Purchases: {len(purchases)} items ({', '.join(set(categories))})")
    except:
        print(f"  Purchases: 0 items")
    
    # Show review preview
    if row['Product Reviews']:
        review_preview = row['Product Reviews'][:50] + "..." if len(row['Product Reviews']) > 50 else row['Product Reviews']
        print(f"  Review: {review_preview}")

print("\n" + "=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)
print(f"✓ No missing values in key columns: {df[['Age', 'Annual Income', 'Time on Site']].isnull().sum().sum() == 0}")
print(f"✓ Income range: ${df['Annual Income'].min():,} to ${df['Annual Income'].max():,}")
print(f"✓ Age range: {df['Age'].min()} to {df['Age'].max()} years")
print(f"✓ Time on Site range: {df['Time on Site'].min():.1f} to {df['Time on Site'].max():.1f} minutes")

print(f"\n📊 JSON FIELDS STATUS:")
print("-" * 40)
purchase_counts = []
for history in df['Purchase History']:
    try:
        purchases = json.loads(history)
        purchase_counts.append(len(purchases))
    except:
        purchase_counts.append(0)

print(f"Average purchases per customer: {np.mean(purchase_counts):.1f}")
print(f"Customers with purchases: {sum(1 for x in purchase_counts if x > 0)}")
print(f"Customers with reviews: {df['Product Reviews'].str.len().gt(0).sum()}")

print("\n" + "=" * 70)
print("✅ READY FOR YOUR STREAMLIT DASHBOARD")
print("=" * 70)
print("\nWhat you'll see in each section:")

print("\n📊 DATA OVERVIEW:")
print("  • 1000 customers with complete profiles")
print("  • Strong Income-Time correlation (~0.65)")
print("  • Moderate Age-Income correlation (~0.55)")

print("\n🔍 FEATURE ENGINEERING:")
print("  • JSON purchase/browsing history for extraction")
print("  • Text reviews for sentiment analysis")
print("  • All original columns preserved")

print("\n🧠 NEURAL NETWORKS:")
print("  • Time on Site strongly predicts Income")
print("  • Good features for classification/regression")
print("  • Realistic patterns for model learning")

print("\n🔬 PATTERN RECOGNITION:")
print("  • Clear correlation patterns")
print("  • Natural customer segments")
print("  • Discoverable relationships")

print("\n📈 INSIGHTS:")
print("  • Higher income = more time on site")
print("  • Older customers = higher income")
print("  • Income affects purchase behavior")

GENERATING DATASET WITH ORIGINAL STRUCTURE

📊 CORRELATIONS IN NUMERIC FEATURES:
--------------------------------------------------

Correlation Matrix:
                         Age  Annual Income  Time on Site  \
Age                    1.000          0.529         0.618   
Annual Income          0.529          1.000         0.854   
Time on Site           0.618          0.854         1.000   
Total_Purchase_Amount  0.374          0.526         0.540   

                       Total_Purchase_Amount  
Age                                    0.374  
Annual Income                          0.526  
Time on Site                           0.540  
Total_Purchase_Amount                  1.000  

Key Relationships:
------------------------------
Annual Income ↔ Time on Site: 0.854
Age ↔ Annual Income: 0.529
Annual Income ↔ Purchase Amount: 0.526

DATASET SUMMARY

✅ File saved: E-commerce_original.csv
📊 Shape: (1000, 9)
📋 Columns: Customer ID, Age, Gender, Location, Annual Income, Purchase History,